# 01 — Collect a Training Dataset

This notebook collects transition data from Procedural Frozen Lake and pushes it to the Hugging Face Hub so the same rollout data can be reused across training runs.

In [ ]:
import torch

from mouse_envs import EnvConfig, make_group_env
from mouse_core.data import Datastore, push_stores_to_hub


DATASET_ID  = "mouse-example-dataset"  # Hub repo id for push_stores_to_hub
TOTAL_STEPS = 1500                       # env steps to collect (one step = one GroupEnv.step call)
NUM_ENVS    = 1000                       # parallel Procedural Frozen Lake instances (one Datastore each)

## Build Environment

`EnvConfig` describes the environment you want to collect from. `make_group_env` builds a `GroupEnv` that steps all instances in parallel. See the `mouse-env` package for more information about available environments and configuration options.

Key parameters for this run:

- **One config per env instance** - `NUM_ENVS` Procedural Frozen Lake env instances are listed explicitly in `configs`. To add or remove envs, edit `NUM_ENVS`.
- **`reset_seed=i`** - each env instance gets its own reset stream.
- **`map_seed=i`** - each env instance gets a deterministic Procedural Frozen Lake map.
- **`episodes_per_task=0`** - each env instance runs one infinite task, so episode boundaries do not reset the policy context.
- **`is_slippery=False`** - actions are deterministic, which keeps the oracle labels and learned behavior easy to inspect.
- **`min_width=4, max_width=8, min_height=4, max_height=8`** - maps can be as large as 8x8, matching the 64-observation vocab used by the training examples.
- **`max_episode_steps=50`** - episodes truncate after 50 steps if the agent has not reached a terminal state.
- **`q_star_source={"provider": "env_q_star"}`** - runs value iteration for the current map and attaches the exact optimal Q-values to every step output as `info_q_star`.

In [ ]:
configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"proc_frozenlake_{i}",
        reset_seed=i,
        episodes_per_task=0,
        kwargs={
            "is_slippery": False,
            "min_width": 4, "max_width": 8,
            "min_height": 4, "max_height": 8,
            "step_penalty": -0.01,
            "max_episode_steps": 50,
            "map_seed": i,
        },
        q_star_source={"provider": "env_q_star"},  # attached to every step as info_q_star
    )
    for i in range(NUM_ENVS)
]

env = make_group_env(configs)
print(f"Environments {min(10, len(env.names))} of {len(env.names)}:")
for n in env.names[:10]:
    print(n)

## Create Datastores

A `Datastore` is a sequential container for step data backed by a Hugging Face Arrow dataset. Each row is a plain Python dict — no fixed schema required. Whatever fields your environment returns, you can store them as-is.

The `name` you give a store becomes its config identifier on the Hub, so each parallel env instance can be fetched back by name.

In [ ]:
stores = [Datastore(name=name) for name in env.names]  # one store per parallel env instance

## Collect transitions

Each step merges the input we sent (`{"action": ...}`) with the environment's output (`observation`, `reward`, `done`, `info_q_star`, ...) into a single dict and appends it to the matching store.

**Oracle ramp:** at step `t`, `oracle_prob = t / (TOTAL_STEPS - 1)`. Each env instance independently samples: with probability `oracle_prob` it picks `argmax(info_q_star)` (the optimal action); otherwise it picks a random action. This means the first steps explore freely while later steps demonstrate expert behavior.

**Initial reset frame:** the very first `env.step` triggers the environment reset; the env ignores the action and returns the initial observation. We store this frame so every datastore starts from timestep 0 and is time-aligned.

In [ ]:
def pick_inputs(outputs, env, oracle_prob: float) -> list[dict]:
    random_inputs = env.sample_random_input()
    inputs = []
    for i, out in enumerate(outputs):
        if torch.rand(1).item() >= oracle_prob:  # explore
            inputs.append(random_inputs[i])
        else:
            action = out.get("info_q_star").argmax()  # best action per Q*
            inputs.append({"action": torch.tensor(action, dtype=torch.int64)})
    return inputs

outputs = None
env.tracker.clear()
for t in range(TOTAL_STEPS):
    oracle_prob = t / (TOTAL_STEPS - 1)         # 0.0 → 1.0 over the run
    if outputs is None:                          # first step triggers reset
        inputs = env.sample_random_input()
    else:
        inputs = pick_inputs(outputs, env, oracle_prob)
    outputs = env.step(inputs)
    for store, inp, out in zip(stores, inputs, outputs):
        store.append({**inp, **out})             # merge input + output into one row

env.close()

for name, rewards, lengths in zip(env.names, env.tracker.episode_cum_rewards, env.tracker.episode_lengths):
    n = len(rewards)
    mean_r = torch.tensor(rewards).mean().item() if n else float("nan")
    mean_l = torch.tensor(lengths).mean().item() if n else float("nan")
    print(f"{name}: {n} episodes | mean reward {mean_r:.3f} | mean length {mean_l:.1f}")

## Push to the Hub

Upload all stores to your Hugging Face account as a single dataset repo. Each store is saved as a separate named config, which keeps the parallel env streams distinct when the dataset is loaded later.

In [ ]:
url = push_stores_to_hub(stores, repo_id=DATASET_ID, split="train", private=False)
print(f"Pushed to {url}")